# ميزان الذهب — تشغيل المشروع كاملًا على Google Colab

شغّل الخلايا من الأعلى إلى الأسفل: **تنزيل → تثبيت معزول → تحقق → تشغيل → رابط الواجهة**.
لا تحتاج إلى إعادة تدريب أو GPU. أحدث نموذج مرفق يدمج ResNet18 وEfficientNet-B0 مع عتبة 55%.
نتيجته المسجلة **88.45% على اختبار المشروع**؛ لم يصل إلى هدف 90%، ولا يثبت أصالة الذهب أو عياره.

- مصدر البرنامج: https://github.com/Aishaas1/AI-porject-new
- للواجهة الخارجية تحتاج حساب ngrok ورمزه، ويُدخل في حقل مخفي أو من أسرار Colab.
- يمكن اختبار صورة داخل الدفتر دون ngrok أو رابط عام.
- خانات رفع الصور وإعادة التدريب والإيقاف معطلة افتراضيًا؛ تفعيلها قرار اختياري.
- يتطلب تنزيل المستودع والأوزان والمكتبات اتصال الإنترنت. انتهاء جلسة Colab يوقف الخدمة؛ احتفظ بالدفتر والنتائج المهمة.


## 1. الإعدادات
اترك القيم الافتراضية لتشغيل الواجهة. لا تضع أي رمز أو كلمة مرور داخل هذا الدفتر.
لإلغاء الرابط العام غيّر `OPEN_PUBLIC_UI` إلى `False`. إذا كان المستودع خاصًا فوفّر وصول GitHub الآمن مسبقًا؛ لا تضع رمز GitHub في رابط التنزيل.


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, time, hashlib, urllib.request, urllib.error

REPOSITORY_URL = "https://github.com/Aishaas1/AI-porject-new.git"
REPO = Path("/content/AI-porject-new")
RUNTIME = Path("/content/gold-runtime")
RUN_DIR = Path("/content/gold-run")
MODEL_NAME = "gold_selected.pt"
PORT = 5000
OPEN_PUBLIC_UI = True

print("إصدار Python في Colab:", sys.version.split()[0])
print("الاستدلال سيعمل على CPU؛ لا يلزم تفعيل GPU.")


## 2. تنزيل المشروع
تُستخدم النسخة الموجودة إذا نُفذت الخلية سابقًا؛ لا تُحذف تعديلاتك ولا يحدث تحديث تلقائي.
لتحديث نسخة سابقة يدويًا: نفّذ `git pull origin main` داخل المجلد بعد حفظ تعديلاتك.


In [ ]:
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPO)], check=True)
else:
    if not (REPO / ".git").exists():
        raise RuntimeError("المجلد موجود لكنه ليس المستودع المتوقع. اختر مسارًا جديدًا في الإعدادات.")
    origin = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if origin.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("المجلد يخص مستودعًا مختلفًا؛ لن أستبدله.")
    print("استخدام النسخة الموجودة؛ لم تُستبدل ملفاتها.")

PROJECT_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("نسخة المشروع:", PROJECT_COMMIT)
assert (REPO / "app.py").is_file(), "ملف التطبيق غير موجود"
RUN_DIR.mkdir(parents=True, exist_ok=True)


## 3. تثبيت المكتبات في بيئة معزولة
لا نستبدل PyTorch أو NumPy في جلسة Colab الرئيسية؛ يستخدم التطبيق مفسرًا مستقلًا.
إذا كان Python الرئيسي خارج 3.10–3.12، تُستخدم أداة uv لإنشاء بيئة Python 3.12.
قد يستغرق التثبيت الأول عدة دقائق. لا تُعطّل التحقق من TLS أو بصمات التنزيل عند حدوث خطأ.


In [ ]:
if not (RUNTIME / "bin" / "python").exists():
    if (3, 10) <= sys.version_info[:2] <= (3, 12):
        subprocess.run([sys.executable, "-m", "venv", str(RUNTIME)], check=True)
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--seed", "--python", "3.12", str(RUNTIME)], check=True)

RUNTIME_PY = str(RUNTIME / "bin" / "python")
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV.update(KERAS_BACKEND="torch", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")
subprocess.run([RUNTIME_PY, "-m", "pip", "install", "torch==2.5.1", "torchvision==0.20.1",
                "--index-url", "https://download.pytorch.org/whl/cpu"], check=True, env=RUNTIME_ENV)
subprocess.run([RUNTIME_PY, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True, env=RUNTIME_ENV)
subprocess.run([RUNTIME_PY, "-m", "pip", "check"], check=True, env=RUNTIME_ENV)
subprocess.run([RUNTIME_PY, "-c", "import torch, torchvision, flask, keras; print('Runtime ready:', torch.__version__, torchvision.__version__)"],
               check=True, env=RUNTIME_ENV)
print("البيئة جاهزة؛ لا يلزم إعادة تشغيل جلسة Colab الرئيسية.")


## 4. التحقق من الأوزان
تُقارن بصمة النموذج بما هو محفوظ مع المشروع، ثم تُختبر بنية النموذج دون تنزيل ImageNet مجددًا.
أي اختلاف يوقف التشغيل بدل استعمال ملف غير متوافق.


In [ ]:
MODEL_PATH = REPO / "artifacts" / MODEL_NAME
if not MODEL_PATH.is_file():
    raise FileNotFoundError("الأوزان المرفقة غير موجودة. تأكد من تنزيل آخر نسخة من المستودع.")
selection = json.loads((REPO / "reports" / "candidate_selection.json").read_text())
actual_hash = hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest()
if actual_hash != selection["checkpoint_sha256"]:
    raise RuntimeError("بصمة الأوزان تختلف عن تقرير الاختيار؛ افحص النسخة قبل تشغيلها.")

check_model = """
import json, sys
from model import load_model
model = load_model(sys.argv[1])
print(json.dumps({'architecture': model.architecture, 'preprocessing': model.preprocessing,
                  'decision_threshold': model.decision_threshold}))
"""
verified = subprocess.check_output([RUNTIME_PY, "-c", check_model, str(MODEL_PATH)],
                                   cwd=REPO, env=RUNTIME_ENV, text=True)
print("بصمة النموذج:", actual_hash)
print("إعدادات النموذج:", verified.strip())


## 5. تشغيل الخادم والتحقق من جاهزيته
تكرار هذه الخلية يوقف الخادم السابق الذي بدأه هذا الدفتر فقط. لا يُوقف أي برنامج آخر يشغل المنفذ.
سجل الأخطاء محفوظ في مجلد التشغيل خارج المستودع.


In [ ]:
import socket
if "tunnel" in globals() and tunnel is not None:
    from pyngrok import ngrok
    ngrok.disconnect(tunnel.public_url)
    tunnel = None
if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=15)
if "server_log" in globals() and not server_log.closed:
    server_log.close()

with socket.socket() as probe:
    probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        probe.bind(("127.0.0.1", PORT))
    except OSError as error:
        raise RuntimeError("المنفذ مشغول. غيّر PORT أو أوقف البرنامج الذي تملكه، ثم أعد تشغيل هذه الخلية.") from error

SERVER_ENV = RUNTIME_ENV.copy()
SERVER_ENV.update(GOLD_MODEL_PATH=str(MODEL_PATH), GOLD_PORT=str(PORT))
LOG_PATH = RUN_DIR / "app.log"
server_log = LOG_PATH.open("w")
launch = "import os; from app import create_app; create_app().run(host='0.0.0.0', port=int(os.environ['GOLD_PORT']), debug=False, use_reloader=False)"
server = subprocess.Popen([RUNTIME_PY, "-u", "-c", launch], cwd=REPO, env=SERVER_ENV,
                          stdout=server_log, stderr=subprocess.STDOUT)
BASE_URL = f"http://127.0.0.1:{PORT}"
ready = False
for _ in range(120):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen(BASE_URL + "/health", timeout=3) as response:
            health = json.load(response)
        if health.get("model_ready") and health.get("checkpoint") == MODEL_PATH.name:
            ready = True
            break
    except (urllib.error.URLError, TimeoutError, OSError):
        pass
    time.sleep(1)
if not ready:
    if server.poll() is None:
        server.terminate()
        server.wait(timeout=15)
    server_log.close()
    print(LOG_PATH.read_text(errors="replace")[-12000:])
    raise RuntimeError("لم يصبح التطبيق جاهزًا. راجع السجل قبل المتابعة.")
print("الخادم جاهز:", health)


## 6. رابط الواجهة العربية — ngrok
احصل على Authtoken من https://dashboard.ngrok.com/get-started/your-authtoken.
يمكن حفظه في **أسرار Colab** باسم `NGROK_AUTHTOKEN` مع السماح للدفتر بالوصول، أو إدخاله في حقل مخفي.
لا تكتبه في الكود ولا تشاركه هنا. الرابط متاح لمن يملكه؛ الواجهة لا تحتوي تسجيل دخول.
إذا لم ترغب في رابط عام، اجعل `OPEN_PUBLIC_UI=False` واستخدم اختبار الصورة في الخلية التالية.


In [ ]:
if OPEN_PUBLIC_UI:
    import importlib.util
    if importlib.util.find_spec("pyngrok") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "pyngrok>=7,<8"], check=True)
    from pyngrok import ngrok
    from getpass import getpass
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("NGROK_AUTHTOKEN")
    except Exception:
        pass
    if not token:
        token = getpass("أدخل ngrok Authtoken (مخفي): ").strip()
    if not token:
        raise ValueError("يلزم رمز ngrok لفتح الرابط العام؛ يمكن تعطيله في الإعدادات.")
    try:
        ngrok.set_auth_token(token)
    except Exception:
        raise RuntimeError("تعذر إعداد رمز ngrok؛ افحص الرمز دون مشاركته أو طباعته.") from None
    finally:
        del token
    if "tunnel" in globals() and tunnel is not None:
        ngrok.disconnect(tunnel.public_url)
    try:
        tunnel = ngrok.connect(PORT, "http")
    except Exception:
        raise RuntimeError("تعذر فتح نفق ngrok؛ تحقق من الرمز وحدود الحساب. الخادم المحلي ما زال يعمل.") from None
    print("افتح الواجهة:", tunnel.public_url)
    print("إذا ظهرت صفحة تنبيه ngrok، اختر المتابعة إلى الموقع.")
else:
    print("الرابط العام معطل؛ يمكنك اختبار صورة داخل الدفتر أدناه.")


## 7. اختبار صورة داخل الدفتر — اختياري
غيّر `UPLOAD_IMAGE=True` في هذه الخلية ثم شغّلها. يعمل الاختبار دون ngrok، لكنه يحتاج الخادم جاهزًا.
الصورة لا تُرسل إلى GitHub. قد تحفظ أداة رفع Colab نسخة في مساحة الجلسة المؤقتة.
الدرجات غير معايرة؛ التصنيف «حقيقي» يتطلب تجاوز عتبة 55%، وليس مجرد الدرجة الأعلى.


In [ ]:
# غيّر إلى True عندما تريد اختيار صورة.
UPLOAD_IMAGE = False
if UPLOAD_IMAGE:
    from google.colab import files
    import uuid
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("لم تُحدد صورة.")
    name, image_bytes = next(iter(uploaded.items()))
    if len(image_bytes) > 8 * 1024 * 1024:
        raise ValueError("الصورة أكبر من 8 ميجابايت.")
    boundary = "GoldNotebook" + uuid.uuid4().hex
    body = (f'--{boundary}\r\nContent-Disposition: form-data; name="image"; filename="upload.img"\r\n'
            f'Content-Type: application/octet-stream\r\n\r\n').encode() + image_bytes + f'\r\n--{boundary}--\r\n'.encode()
    request = urllib.request.Request(BASE_URL + "/predict", data=body,
                                    headers={"Content-Type": f"multipart/form-data; boundary={boundary}"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            result = json.load(response)
    except urllib.error.HTTPError as error:
        print(error.read().decode(errors="replace"))
        raise
    from IPython.display import display, Image as DisplayImage
    display(DisplayImage(data=image_bytes, width=360))
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print("اختبار رفع الصورة اختياري؛ فعّله عند الحاجة.")


## 8. إعادة التدريب من البيانات الأصلية — اختياري ومكلف
التشغيل العادي مكتمل دون هذه الخطوة. غيّر `RUN_TRAINING=True` في الخلية التالية فقط إذا أردت إعادة تجربة التدريب.
تُستخدم نسخة تدريب مستقلة حتى لا تتغير ملفات التطبيق العامل أو أوزانه المرفقة.
تُنزّل GoldNet من الإصدار المحدد وتُحفظ الاستبعادات الخمسة؛ لا تُضاف صور ذهب جديدة إلى التجربة.

هذه البرامج تعمل على CPU حتى إن فعّلت GPU في Colab، وقد تستغرق وقتًا طويلًا وتتوقف بانتهاء الجلسة.
إعادة التنفيذ لا تضمن النتيجة نفسها أو الوصول إلى 90%. لا تغيّر الإعدادات بعد قراءة الاختبار لمحاولة رفع رقمه.


In [ ]:
RUN_TRAINING = False
TRAIN_REPO = Path("/content/gold-training")
GOLDNET = Path("/content/GoldNet-data-source")
GOLDNET_COMMIT = "f206c941a39d3126b6ce867e01abfad87e10f525"
if RUN_TRAINING:
    if TRAIN_REPO.exists():
        raise RuntimeError("مجلد التدريب موجود. افحصه أو اختر مسارًا جديدًا؛ لن يُستبدل تلقائيًا.")
    subprocess.run(["git", "clone", "--local", "--no-hardlinks", str(REPO), str(TRAIN_REPO)], check=True)
    if not GOLDNET.exists():
        subprocess.run(["git", "clone", "https://github.com/zobeirraisi/GoldNet.git", str(GOLDNET)], check=True)
        subprocess.run(["git", "-C", str(GOLDNET), "checkout", GOLDNET_COMMIT], check=True)
    actual = subprocess.check_output(["git", "-C", str(GOLDNET), "rev-parse", "HEAD"], text=True).strip()
    if actual != GOLDNET_COMMIT:
        raise RuntimeError("نسخة GoldNet لا تطابق الإصدار المطلوب؛ لن أغيّر ملفات موجودة تلقائيًا.")
    commands = [
        ["prepare_goldnet.py", "--source", str(GOLDNET), "--destination", "data/goldnet"],
        ["audit_data.py", "--data", "data/goldnet"],
        ["train_efficient.py"],
        ["adapt_bn.py"],
        ["select_candidate.py"],
    ]
    for arguments in commands:
        print("تشغيل:", arguments[0], flush=True)
        subprocess.run([RUNTIME_PY] + arguments, cwd=TRAIN_REPO, env=RUNTIME_ENV, check=True)
    print("انتهى التدريب والاختيار على التحقق. الأوزان في:", TRAIN_REPO / "artifacts/gold_selected.pt")
    print("الواجهة الحالية ما زالت تستخدم الأوزان الأصلية. لتجربة الجديدة غيّر MODEL_PATH وأعد خلية الخادم.")
else:
    print("إعادة التدريب معطلة؛ الواجهة تستخدم الأوزان المرفقة.")


## 9. تقييم التدريب وحفظ النتائج — اختياري
فعّل هذا القسم بعد اكتمال إعادة التدريب فقط. يقيّم المرشح المختار على جميع صور الاختبار الـ303 دون إسقاط حالات.
الاختبار سبق استخدامه في التطوير، فلا يمثل اختبارًا خارجيًا أعمى. تنزيل الحزمة يتضمن الأوزان والتقارير فقط، ولا يشمل صور البيانات.


In [ ]:
RUN_TEST_EVALUATION = False
if RUN_TEST_EVALUATION:
    if not (TRAIN_REPO / "data/goldnet/test").is_dir():
        raise RuntimeError("أكمل قسم التدريب أولًا؛ بيانات الاختبار غير موجودة في نسخة التدريب.")
    subprocess.run([RUNTIME_PY, "evaluate.py", "--data", "data/goldnet/test",
                    "--model", "artifacts/gold_selected.pt", "--output", "reports/selected_test_metrics.json"],
                   cwd=TRAIN_REPO, env=RUNTIME_ENV, check=True)
    import zipfile
    package = RUN_DIR / "gold_training_results.zip"
    with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in (TRAIN_REPO / "reports").glob("*"):
            if path.is_file(): archive.write(path, str(path.relative_to(TRAIN_REPO)))
        for filename in ("gold_selected.pt", "gold_selected.json"):
            path = TRAIN_REPO / "artifacts" / filename
            archive.write(path, "artifacts/" + filename)
    from google.colab import files
    files.download(str(package))
else:
    print("التقييم وإعادة تنزيل النتائج معطلان افتراضيًا.")


## 10. إيقاف الخادم والنفق
هذا القسم لا يوقف البرنامج تلقائيًا عند «تشغيل الكل». غيّر `STOP_APP=True` هنا ونفّذه عند الانتهاء.
لإعادة التشغيل لاحقًا، أعد خلايا الخادم والرابط.


In [ ]:
STOP_APP = False
if STOP_APP:
    if "tunnel" in globals() and tunnel is not None:
        from pyngrok import ngrok
        ngrok.disconnect(tunnel.public_url)
        tunnel = None
    if "server" in globals() and server.poll() is None:
        server.terminate()
        server.wait(timeout=15)
    if "server_log" in globals() and not server_log.closed:
        server_log.close()
    print("أُوقفت الخدمات التي شغّلها هذا الدفتر.")
else:
    print("التطبيق مستمر. فعّل STOP_APP عندما تنتهي.")


## حل المشكلات

| الحالة | الإجراء |
| --- | --- |
| خطأ تثبيت | افحص آخر سطر من الخطأ؛ استخدم جلسة جديدة وأعد الخلايا بالترتيب. لا تعطل TLS. |
| الأوزان أو البصمة مختلفة | احفظ تعديلاتك ثم نزّل نسخة جديدة في مسار جديد؛ لا تتجاوز فحص البصمة. |
| المنفذ مشغول | أوقف خادم هذا الدفتر أو اختر PORT آخر، ثم أعد تشغيل الخادم والنفق. |
| خطأ ngrok | تحقق من رمز حسابك وحدود خطتك؛ الاختبار داخل الدفتر يعمل دون نفق. |
| الرابط توقف | تحقق أن جلسة Colab والخادم ما زالا يعملان؛ أعد خلية النفق إن لزم. |
| فشل الصورة | JPG/PNG/WebP سليمة، وطلب أقل من 8 ميجابايت، وأبعاد حتى 20 مليون بكسل. |
| إعادة التدريب توقفت | تتوقف العمليات بانتهاء جلسة Colab؛ احفظ المخرجات المهمة قبل الانتهاء. |

هذا الدفتر لا يرفع الصور أو النتائج تلقائيًا إلى GitHub، ولا يطلب منك نشر أي مفاتيح.
لا توجد خدمة استضافة دائمة أو ضمان نسبة 90%. راجع README وتقارير المشروع لحدود التقييم.
